In [122]:
%load_ext autoreload
%autoreload 2

import os, glob
import numpy as np
import torch
import torch.nn as nn
import torchaudio.transforms as T
from src.constants import Constants as C
from pathlib import Path

from src.parsers import PhonemeWindowDataset
from src.NeuralModel import CRNN
from src.trainers import train_model, evaluate_tm, load_checkpoint
from src.evaluator import evaluate_audio
from src.wordmaker import PHONEME_TO_LETTERS, levenshtein_distance, phonemes_to_text, parse_words, WLIST1000, proba_predict

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [123]:
CHECKPOINT_PATH = "../trained_models/BetterDataSoft.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CRNN()
meta = load_checkpoint(CHECKPOINT_PATH, model, device=device)
model.eval()

print("checkpoint meta keys:", list(meta.keys()))

checkpoint meta keys: []


In [124]:
from src.wordmaker import dictionary_extend
null_dict = []
dictionary_extend(null_dict, "../AutorskieDane/AutorskiDataset")


znaleziono 31 plików


['kordian',
 'i',
 'laura',
 'spacerują',
 'po',
 'ogrodzie',
 'ukochana',
 'bohatera',
 'jest',
 'od',
 'niego',
 'nieco',
 'starsza',
 'co',
 'widać',
 'również',
 'w',
 'sposobie',
 'postrzegania',
 'przez',
 'nią',
 'rzeczywistości',
 'prezentuje',
 'się',
 'jako',
 'zakochany',
 'uszy',
 'młodzieniec',
 'który',
 'pała',
 'do',
 'laury',
 'szczerym',
 'autentycznie',
 'gorącym',
 'uczuciem',
 'dziewczyna',
 'traktuje',
 'go',
 'jednak',
 'chłodniej',
 'trzyma',
 'trochę',
 'na',
 'dystans',
 'zdaje',
 'momentami',
 'że',
 'bliżej',
 'jej',
 'traktowania',
 'młodszego',
 'brata',
 'niż',
 'przyszłego',
 'kochanka',
 'przeczytaniu',
 'listu',
 'zostawił',
 'pamiętniku',
 'domyśla',
 'chłopak',
 'planuje',
 'samobójstwo',
 'karci',
 'za',
 'takie',
 'myśli',
 'wypowiedzi',
 'postrzega',
 'wyraz',
 'matczynej',
 'czy',
 'siostrzanej',
 'troski',
 'a',
 'nie',
 'głębokiego',
 'uczucia',
 'kobiety',
 'mężczyzny',
 'to',
 'jeszcze',
 'bardziej',
 'upewnia',
 'podjętym',
 'zamiarze',
 'bo

In [125]:
from src.levenshtein import lev_weighted
from src.levenshtein import damerau_lev
from src.levenshtein import true_damerau_levenshtein
from src.levenshtein import damerau_levenshtein_weighted
from src.levenshtein import levenshtein_phoneme_aware
from src.levenshtein import damerau_levenshtein_neighbour_aware
from src.constants import Constants as C
from src.wordmaker import folder_search_accuracy

In [126]:
datdir = '../AutorskieDane/AutorskiDataset/'

In [15]:
acc = folder_search_accuracy(data_dir=datdir, dictionary=null_dict, lev=damerau_levenshtein_neighbour_aware)
print(acc) #zwykly substitution cost

znaleziono 31 plików


/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:42: WavFileWarning: Chunk (non-data) not understood, skipping it.
  samplerate, audio = wavfile.read(wav_path)


0.463519313304721


In [16]:
acc = folder_search_accuracy(data_dir=datdir, dictionary=null_dict, lev=damerau_levenshtein_weighted)
print(acc) #zwykly substitution cost

znaleziono 31 plików
0.49908031882280807


In [127]:
acc = folder_search_accuracy(model=model, data_dir=datdir, dictionary=null_dict, lev=damerau_levenshtein_weighted)
print(acc)

found 31 files, search works
0.5346413243408952


In [69]:
datdir = '../AutorskieDane/AutorskiDataset/'
acc = folder_search_accuracy(data_dir=datdir, dictionary=null_dict, lev=lev_weighted)
print(acc)

znaleziono 31 plików
0.4819129368485592


In [70]:
acc = folder_search_accuracy(data_dir=datdir, dictionary=null_dict, lev=levenshtein_distance)
print(acc)

znaleziono 31 plików
0.3893316983445739


In [ ]:
datdir = '../AutorskieDane/AutorskiDataset/'
acc = folder_search_accuracy(data_dir=datdir, dictionary=null_dict, lev=damerau_lev)
print(acc)

znaleziono 31 plików
0.38197424892703863


In [ ]:
acc = folder_search_accuracy(data_dir=datdir, dictionary=null_dict, lev=true_damerau_levenshtein)
print(acc)

znaleziono 31 plików
0.38136112814224404


In [ ]:

"""

from src.wordmaker import mel_cut
from src.wordmaker import predict_from_exact_mel
from src.parsers import wav_to_logmel
from src.evaluator import evaluate_word

def mel_cut(word_info, mel): #(start_time, end_time, word)
    hop_time = C.FRAME_MS / 1000
    n_start = int(word_info[0] / hop_time)
    n_end = int(word_info[1] / hop_time)
    mel_exact = mel[:,n_start:n_end]
    return mel_exact

def predict_from_exact_mel(target_word, mel_exact, dictionary, model, lev = lev_weighted, verbose=False, proba_threshold=0.67, top_phonemes=4):
    result = evaluate_word(mel=mel_exact, model=model, top_k=top_phonemes)
    w = proba_predict(result, p=proba_threshold, longer_reg = True, aeo_reg=True, verbose=False)
    wtext = phonemes_to_text(w, after_silence=False)
    output = dictionary[0]
    mindist = 1000
    for wr in dictionary:
        #print(f"Testing word: {wr}")
        #print(f'against word: {w}')
        dist = lev(wr, wtext)
        if dist < mindist:
            mindist = dist
            output = wr
    if(verbose):
        print(f'word from model: {wtext}')
        print(f'output: {output}')
        print(f'target: {target_word}')
    return output


def folder_search_accuracy(data_dir, dictionary, lev):
    correct = 0
    incorrect = 0 
    tg_paths = sorted(str(p) for p in Path(data_dir).rglob("*.TextGrid"))
    print(f"found {len(tg_paths)} files, search works")

    for tg in tg_paths:

        wav_path = tg[: -len(".TextGrid")] + ".wav"
        PATH = Path(tg)
        with open(PATH, "r", encoding="utf-8") as f:
            words_timeframes = parse_words(f.read())
        mel = wav_to_logmel(wav_path=wav_path)

        for w_info in words_timeframes:
            if(w_info[2] != 'sil' and (w_info[1] - w_info[0]) > C.WIN_MS/1000):
                mel_exact = mel_cut(w_info, mel)
                output = predict_from_exact_mel(w_info[2], mel_exact, dictionary, model, lev=lev, verbose=False)
                if output == w_info[2]:
                    correct += 1
                else:
                    incorrect += 1
                    
    return correct / (correct+incorrect)
"""